# Mini-Pilot Interface Study A (without Practice)
(Testing the interface's functionality)

In [ ]:
# Mini-Pilot Study A (NO PRACTICE)
# Study A Interface – Gradio 5.49.1
# Flow: Welcome → Instructions → Main → End
# All core functions as in Main Study Code: slot assignment, IMC global + embedded, GOLD, min-time, CSV append, UI/scroll hint

# Setup (Gradio und Pandas)
!pip install gradio==5.49.1 pandas --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import gradio as gr
import pandas as pd
import glob
import os, re, time, random, math, logging

logger = logging.getLogger("studyA")
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler())

Mounted at /content/drive


## Load Data for Mini-Pilot

In [ ]:
# INPUT: final Stimuli
STIMULI_FULL = "/content/drive/MyDrive/Masterarbeit/Dialoge/studyA_dialogs_from_goodbad.csv"

# OUTPUT: Mini-Pilot Data
PILOT_DIR = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA_MiniPilot"
os.makedirs(PILOT_DIR, exist_ok=True)

STIMULI_CSV = os.path.join(PILOT_DIR, "studyA_stimuli_MINIPILOT.csv")
ASSIGN_CSV  = os.path.join(PILOT_DIR, "studyA_assign_MINIPILOT.csv")

# RESULTS DIR (for CSV-Append)
RESULTS_DIR = os.path.join(PILOT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# Control size of the pilot
N_GOOD = 6
N_BAD  = 6
N_GOLD_GOOD = 1
N_GOLD_BAD  = 1
N_IMC_GOOD  = 1
N_IMC_BAD   = 1

df = pd.read_csv(STIMULI_FULL)

# Safetycheck
needed = {"dialog_id","recipe_title","condition","dialog_text","is_gold","gold_expected_min","gold_expected_max","is_imc","imc_expected_overall"}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in STIMULI_FULL: {missing}")

# Part Pool
df_good = df[df["condition"] == "good"].copy()
df_bad  = df[df["condition"] == "bad"].copy()

# Targeted intake of gold/IMC
gold_good = df_good[df_good["is_gold"] == 1].head(N_GOLD_GOOD)
gold_bad  = df_bad[df_bad["is_gold"] == 1].head(N_GOLD_BAD)

imc_good  = df_good[df_good["is_imc"] == 1].head(N_IMC_GOOD)
imc_bad   = df_bad[df_bad["is_imc"] == 1].head(N_IMC_BAD)

# Fill the rest randomly (without duplicates)
exclude_ids = set(pd.concat([gold_good, gold_bad, imc_good, imc_bad])["dialog_id"].tolist())

rest_good = df_good[~df_good["dialog_id"].isin(exclude_ids)].sample(
    n=max(0, N_GOOD - len(gold_good) - len(imc_good)),
    random_state=42
)
rest_bad = df_bad[~df_bad["dialog_id"].isin(exclude_ids)].sample(
    n=max(0, N_BAD - len(gold_bad) - len(imc_bad)),
    random_state=42
)

df_pilot = pd.concat([gold_good, imc_good, rest_good, gold_bad, imc_bad, rest_bad], ignore_index=True)

# Save
df_pilot.to_csv(STIMULI_CSV, index=False, encoding="utf-8")
print("Saved pilot stimuli:", STIMULI_CSV, "shape=", df_pilot.shape)

Saved pilot stimuli: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyA_MiniPilot/studyA_stimuli_MINIPILOT.csv shape= (12, 9)


In [ ]:
# Create ASSIGN: each dialog 4 slots
assign_df = df_pilot[["dialog_id","recipe_title","condition"]].copy()
assign_df["remaining_slots"] = 4
assign_df.to_csv(ASSIGN_CSV, index=False, encoding="utf-8")
print("Saved pilot assign:", ASSIGN_CSV, "shape=", assign_df.shape)

display(df_pilot[["dialog_id","condition","is_gold","is_imc","gold_expected_min","gold_expected_max","imc_expected_overall"]])

Saved pilot assign: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyA_MiniPilot/studyA_assign_MINIPILOT.csv shape= (12, 4)


,dialog_id,condition,is_gold,is_imc,gold_expected_min,gold_expected_max,imc_expected_overall
0,D001,good,1,0,6.0,7.0,NaN
1,D013,good,0,1,NaN,NaN,7.0
2,D036,good,0,0,NaN,NaN,NaN
3,D039,good,0,0,NaN,NaN,NaN
4,D006,good,0,0,NaN,NaN,NaN
5,D016,good,0,0,NaN,NaN,NaN
6,D043,bad,1,0,1.0,2.0,NaN
7,D051,bad,0,1,NaN,NaN,1.0
8,D076,bad,0,0,NaN,NaN,NaN
9,D079,bad,0,0,NaN,NaN,NaN


## Mini-Pilot (Gradio-Interface Code without Practice)

In [ ]:
# Mini-Pilot (Gradio Interface-Code)

logger.info("Study A – Welcome → Instructions → Main → End (NO PRACTICE)")

# Max. 20 Dialogs in Main
MAX_MAIN_DIALOGS = 20
# Seconds, for min_time_ok_dialog
MIN_DIALOG_TIME = 20.0  # nach Mini-Pilot anpassen

# Load dialog data from STIMULI_CSV
dialogs_df = pd.read_csv(STIMULI_CSV)
logger.info(f"[CORE] dialogs_df loaded, shape={dialogs_df.shape}")

dialogs_list = dialogs_df.to_dict(orient="records")
logger.info(f"[CORE] dialogs_list len={len(dialogs_list)}")

Study A – Welcome → Instructions → Main → End (NO PRACTICE)
INFO:studyA:Study A – Welcome → Instructions → Main → End (NO PRACTICE)
[CORE] dialogs_df loaded, shape=(12, 9)
INFO:studyA:[CORE] dialogs_df loaded, shape=(12, 9)
[CORE] dialogs_list len=12
INFO:studyA:[CORE] dialogs_list len=12


In [ ]:
# Dialog → Bubbles

def parse_dialog_turns(raw: str):
    logger.info(f"[parse_dialog_turns] raw type={type(raw).__name__}")
    if not isinstance(raw, str):
        return []
    text = re.sub(r'\s*(Fat:)', r'\n\1', raw)
    text = re.sub(r'\s*(Carb:)', r'\n\1', text)
    text = text.strip()
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    logger.info(f"[parse_dialog_turns] lines={lines!r}")

    turns = []
    for line in lines:
        if line.startswith("Fat:"):
            speaker = "Fat"
            content = line[len("Fat:"):].strip()
        elif line.startswith("Carb:"):
            speaker = "Carb"
            content = line[len("Carb:"):].strip()
        else:
            speaker = "Other"
            content = line
        turns.append((speaker, content))
    logger.info(f"[parse_dialog_turns] turns={turns!r}")
    return turns


def dialog_to_bubbles(raw: str) -> str:
    turns = parse_dialog_turns(raw)
    if not turns:
        return ""
    html = ["<div class='dialog-container'>"]
    for speaker, content in turns:
        if speaker == "Fat":
            bubble_class = "bubble bubble-fat"
            name = "Fat"
        elif speaker == "Carb":
            bubble_class = "bubble bubble-carb"
            name = "Carb"
        else:
            bubble_class = "bubble bubble-other"
            name = ""
        speaker_html = f"<div class='speaker'>{name}</div>" if name else ""
        html.append(
            f"<div class='{bubble_class}'>{speaker_html}<div class='bubble-text'>{content}</div></div>"
        )
    html.append("</div>")
    html_out = "\n".join(html)
    logger.info(f"[dialog_to_bubbles] html length={len(html_out)}")
    return html_out

In [ ]:
# Subscales & Likert

SUBSCALES = [
    ("truthfulness", "The dialogue is internally consistent, without contradictions or misleading statements."),
    ("relevance", "Each turn in the dialogue responds meaningfully to the previous one and stays on topic."),
    ("clarity", "The dialogue is clear, easy to understand, and expressed in unambiguous language."),
    ("relation_appropriateness", "The dialogue handles the interpersonal relationship appropriately, without tension, dominance, or relational violations."),
    ("logic_coherence", "The dialogue presents coherent reasoning, with arguments that follow a logical structure."),
    ("respect_appreciation", "The dialogue maintains a respectful tone, acknowledging the partner’s contributions appropriately."),
    ("feedback_depth", "The dialogue builds meaningfully on the partner’s statements and contains sufficient conversational depth."),
    ("overall", "Overall, the dialogue shows high communication quality."),
]

LIKERT_LABELS = [1, 2, 3, 4, 5, 6, 7]

def parse_likert(choice):
    logger.info(f"[parse_likert] choice={choice!r}, type={type(choice).__name__}")
    if choice is None or choice == "":
        return None
    try:
        val = int(choice)
        logger.info(f"[parse_likert] parsed={val}")
        return val
    except Exception as e:
        logger.info(f"[parse_likert] parse error: {e!r}")
        return None

def main_progress_text(index, total):
    if total <= 0:
        return ""
    current = min(max(index + 1, 1), total)
    return f"Main: {current} / {total}"

In [ ]:
# Slot-based dialog assignment (ASSIGN_CSV)

def make_main_dialogs_for_rater(n_main=MAX_MAIN_DIALOGS):
    """
    Selects a set of dialogs for a rater that still have slots available,
    updates ASSIGN_CSV (remaining_slots -= 1), and returns a list of records.
    """
    logger.info(f"[make_main_dialogs_for_rater] called with n_main={n_main}")

    # Load Assignment-Data
    try:
        assign_df = pd.read_csv(ASSIGN_CSV)
        logger.info(f"[make_main_dialogs_for_rater] assign_df shape={assign_df.shape}")
    except Exception:
        logger.exception("[make_main_dialogs_for_rater] Could not load ASSIGN_CSV")
        raise

    # Only dialogs with free slots
    available = assign_df[assign_df["remaining_slots"] > 0]
    logger.info(f"[make_main_dialogs_for_rater] available shape={available.shape}")

    if available.empty:
        raise ValueError("No dialogs with remaining slots left in ASSIGN_CSV.")

    # Take recipe title from ASSIGN_CSV (if dialogs_df does not have it)
    merge_cols = ["dialog_id", "remaining_slots"]
    if "recipe_title" in assign_df.columns:
        merge_cols.append("recipe_title")

    # Merge with all dialog data
    merged = dialogs_df.merge(
        available[merge_cols],
        on="dialog_id",
        how="inner",
        suffixes=("", "_assign"),
    )
    logger.info(
        f"[make_main_dialogs_for_rater] merged shape={merged.shape}, "
        f"merged columns={list(merged.columns)}"
    )

    if merged.empty:
        raise ValueError("Merged assignment with dialogs_df is empty.")

    # max. 1 Dialog per recipe
    grouped = merged.groupby("recipe_title", group_keys=False)
    one_per_recipe = grouped.apply(
        lambda g: g.sample(n=1, random_state=random.randint(0, 80)),
        include_groups=False,
    )
    logger.info(f"[make_main_dialogs_for_rater] one_per_recipe shape={one_per_recipe.shape}")

    # Limit to n_main (if available)
    if n_main is not None and n_main < len(one_per_recipe):
        selected = one_per_recipe.sample(
            n=n_main, random_state=random.randint(0, 80)
        )
    else:
        selected = one_per_recipe

    # Shuffle order randomly
    selected = selected.sample(frac=1, random_state=random.randint(0, 80)).reset_index(drop=True)

    # Ensure that a column "recipe_title" exists
    if "recipe_title" not in selected.columns and "recipe_title_assign" in selected.columns:
        selected = selected.rename(columns={"recipe_title_assign": "recipe_title"})

    # If both exist but recipe_title is empty, fill it with recipe_title_assign
    if "recipe_title" in selected.columns and "recipe_title_assign" in selected.columns:
        selected["recipe_title"] = selected["recipe_title"].fillna(selected["recipe_title_assign"])

    logger.info(f"[make_main_dialogs_for_rater] selected shape={selected.shape}")
    used_ids = selected["dialog_id"].tolist()
    logger.info(f"[make_main_dialogs_for_rater] selected dialog_ids={used_ids}")

    # Decrement slots in ASSIGN_CSV
    assign_df.loc[assign_df["dialog_id"].isin(used_ids), "remaining_slots"] -= 1
    assign_df.to_csv(ASSIGN_CSV, index=False)
    logger.info("[make_main_dialogs_for_rater] ASSIGN_CSV updated")

    return selected.to_dict(orient="records")

In [ ]:
# Helper: Show current dialog

def show_current_dialog(main_index, dialogs):
    logger.info(f"[show_current_dialog] main_index={main_index}")
    if dialogs is None or main_index is None:
        return "No dialogs (state is None)."
    if main_index >= len(dialogs):
        return "No more dialogs."
    raw = dialogs[main_index]["dialog_text"]
    return dialog_to_bubbles(raw)

In [ ]:
# CSV append (MAIN)

def append_rating_row(
    rater_id,
    dialog_record,
    ratings,
    comment,
    duration,
    imc_pass_global,
    imc_dialog_pass,
    start_time,
    min_time_ok_dialog,
    gold_pass,
):
    out = os.path.join(RESULTS_DIR, f"ratings_{rater_id}.csv")

    logger.info(f"[append_rating_row] rater_id={rater_id}, dialog_id={dialog_record.get('dialog_id')}")
    logger.info(
        f"[append_rating_row] ratings={ratings}, comment={comment!r}, duration={duration} "
        f"min_time_ok_dialog={min_time_ok_dialog}, gold_pass={gold_pass}"
    )

    row = {
        "start_time": start_time,
        "timestamp": time.time(),
        "rater_id": rater_id,
        "dialog_id": dialog_record.get("dialog_id"),
        "recipe_title": dialog_record.get("recipe_title") or dialog_record.get("recipe"),
        "condition": dialog_record.get("condition"),
        "is_gold": dialog_record.get("is_gold", 0),
        "gold_expected_min": dialog_record.get("gold_expected_min"),
        "gold_expected_max": dialog_record.get("gold_expected_max"),
        "is_imc": dialog_record.get("is_imc", 0),
        "imc_expected_overall": dialog_record.get("imc_expected_overall"),
        "imc_pass_global": imc_pass_global,
        "imc_dialog_pass": imc_dialog_pass,
        "min_time_ok_dialog": min_time_ok_dialog,
        "gold_pass": gold_pass,
        "duration_sec": duration,
        "comment": comment or "",
    }
    for k, v in ratings.items():
        row[k] = v

    df_out = pd.DataFrame([row])
    write_header = not os.path.exists(out)
    logger.info(f"[append_rating_row] write_header={write_header}, out={out}")
    df_out.to_csv(out, mode="a", index=False, header=write_header)
    logger.info(f"[append_rating_row] Row appended to {out}")

In [ ]:
# MAIN Submit callback (20 dialog limit + end screen)

def submit_main_core(
    radio_truth,
    radio_rel,
    radio_clarity,
    radio_relation,
    radio_logic,
    radio_respect,
    radio_feedback,
    radio_overall,
    comment,
    rater_id,
    dialogs,
    main_index,
    dialog_start_time,
    imc_pass_global,
):
    logger.info("=== submit_main_core called ===")
    raw_vals = [
        radio_truth,
        radio_rel,
        radio_clarity,
        radio_relation,
        radio_logic,
        radio_respect,
        radio_feedback,
        radio_overall,
    ]
    logger.info(
        f"[submit_main_core] raw_vals={raw_vals}, comment={comment!r}, "
        f"main_index={main_index}, rater_id={rater_id!r}"
    )

    rating_vals = [parse_likert(v) for v in raw_vals]
    logger.info(f"[submit_main_core] parsed rating_vals={rating_vals}")

    total_dialogs = len(dialogs) if dialogs is not None else 0
    total_main = min(total_dialogs, MAX_MAIN_DIALOGS)
    logger.info(f"[submit_main_core] total_dialogs={total_dialogs}, total_main={total_main}")

    # No or too little dialogue -> directly to end screen
    if dialogs is None or main_index is None or main_index >= total_main or total_main == 0:
        progress_text = main_progress_text(0, total_main)
        debug_msg = f"DEBUG: no more dialogs or none configured. raw={raw_vals}, parsed={rating_vals}, comment={comment!r}"
        logger.info(debug_msg)

        end_text = (
            f"No dialogs available.\n\nYour rater code: **{rater_id}**"
            if rater_id else "No dialogs available."
        )

        return (
            "",                     # status_msg
            gr.update(visible=False),      # dialog_md visible
            main_index or 0,               # main_index_state
            dialog_start_time,             # dialog_start_state
            "No more dialogs.",            # dialog_md value
            True,                          # finished_state
            progress_text,                 # progress_md
            gr.update(visible=False),      # main_group
            gr.update(visible=True),       # end_group
            end_text,                      # end_md
            "",                            # main_title_md
            gr.update(visible=False),      # main_scroll_hint
        )

    current = dialogs[main_index]
    logger.info(f"[submit_main_core] current dialog_id={current.get('dialog_id')}")

    processed_vals = [v if v is not None else 0 for v in rating_vals]
    ratings = {SUBSCALES[i][0]: processed_vals[i] for i in range(len(SUBSCALES))}
    logger.info(f"[submit_main_core] final ratings dict={ratings}")

    duration = time.time() - dialog_start_time if dialog_start_time else None
    logger.info(f"[submit_main_core] duration={duration}")

    # Embedded IMC (dialog level)
    imc_dialog_pass = None
    try:
        if int(current.get("is_imc", 0)) == 1:
            expected = current.get("imc_expected_overall", None)
            try:
                expected_val = int(expected)
            except Exception:
                expected_val = None

            overall_val = ratings.get("overall", None)
            if expected_val is not None and overall_val is not None:
                imc_dialog_pass = int(overall_val == expected_val)
        logger.info(f"[submit_main_core] imc_pass_global={imc_pass_global}, imc_dialog_pass={imc_dialog_pass}")
    except Exception as e:
        logger.exception(f"[submit_main_core] error computing imc_dialog_pass: {e!r}")
        imc_dialog_pass = None

    # GOLD logic (dialog level)
    gold_pass = None
    try:
        if int(current.get("is_gold", 0)) == 1:
            def _to_int_safe(x):
                if x is None:
                    return None
                try:
                    if isinstance(x, float) and math.isnan(x):
                        return None
                except Exception:
                    pass
                try:
                    return int(x)
                except Exception:
                    try:
                        return int(float(str(x)))
                    except Exception:
                        return None

            exp_min = _to_int_safe(current.get("gold_expected_min", None))
            exp_max = _to_int_safe(current.get("gold_expected_max", None))
            overall_val = ratings.get("overall", None)

            if overall_val is not None and exp_min is not None and exp_max is not None:
                gold_pass = int(exp_min <= overall_val <= exp_max)

        logger.info(
            f"[submit_main_core] gold_pass={gold_pass}, "
            f"gold_expected_min={current.get('gold_expected_min')}, "
            f"gold_expected_max={current.get('gold_expected_max')}"
        )
    except Exception as e:
        logger.exception(f"[submit_main_core] error computing gold_pass: {e!r}")
        gold_pass = None

    rated_title = current.get("recipe_title") or current.get("recipe") or ""
    rated_keys = list(current.keys())
    debug_msg = (
        f"DEBUG: raw={raw_vals}, parsed={rating_vals}, comment={comment!r}, "
        f"dialog_id={current.get('dialog_id')}, recipe_title={rated_title}, keys={rated_keys}, "
        f"gold_pass={gold_pass}"
    )
    logger.info(debug_msg)

    # Min-Time flag
    if duration is not None:
        min_time_ok_dialog = int(duration >= MIN_DIALOG_TIME)
    else:
        min_time_ok_dialog = None

    append_rating_row(
        rater_id=rater_id,
        dialog_record=current,
        ratings=ratings,
        comment=comment,
        duration=duration,
        imc_pass_global=imc_pass_global,
        imc_dialog_pass=imc_dialog_pass,
        start_time=dialog_start_time,
        min_time_ok_dialog=min_time_ok_dialog,
        gold_pass=gold_pass,
    )

    # Next dialog
    main_index += 1

    # Done after 20 (or less)
    if main_index >= total_main:
        end_text = (
            f"Thank you for participating!\n\n"
            f"You have completed **{total_main}** dialogues.\n\n"
            f"Your rater code: **{rater_id}**\n\n"
            f"You can now close the window."
            if rater_id else
            f"Thank you for participating!\n\nYou have completed **{total_main}** dialogues.\n\nYou can now close the window."
        )

        return (
            "",                               # status_msg
            gr.update(visible=False),                # dialog_md visible
            main_index,                              # main_index_state
            None,                                    # dialog_start_state
            "No more dialogs to show.",              # dialog_md value
            True,                                    # finished_state
            main_progress_text(total_main - 1, total_main),
            gr.update(visible=False),                # main_group off
            gr.update(visible=True),                 # end_group on
            end_text,                                # end_md
            "",                                      # main_title_md
            gr.update(visible=False),                # main_scroll_hint off
        )

    # Still dialogs left -> show next
    next_text = show_current_dialog(main_index, dialogs)
    new_start_time = time.time()

    next_record = dialogs[main_index]
    main_title = next_record.get("recipe_title") or next_record.get("recipe") or ""

    return (
        "",
        gr.update(visible=True),
        main_index,
        new_start_time,
        next_text,
        False,
        main_progress_text(main_index, total_main),
        gr.update(visible=True),
        gr.update(visible=False),
        "",
        main_title,
        gr.update(visible=(main_index > 0)),
    )

In [ ]:
# Reset controls (MAIN)

def reset_main_controls():
    updates = [gr.update(value=None) for _ in range(len(SUBSCALES))]
    updates.append(gr.update(value=""))
    return updates

In [ ]:
# Welcome & Instructions logic

def toggle_start_button(consent, english_ok, imc_answer):
    enabled = bool(consent and english_ok and imc_answer == "Strongly disagree")
    return gr.update(interactive=enabled)

def toggle_instructions_button(confirmed: bool):
    return gr.update(interactive=bool(confirmed))

def handle_welcome(rater_id_input, english_ok, consent, imc_answer):
    logger.info(
        f"[handle_welcome] rater_id_input={rater_id_input!r}, "
        f"english_ok={english_ok}, consent={consent}, imc_answer={imc_answer!r}"
    )

    # Global IMC flag
    imc_pass_global = int(imc_answer == "Strongly disagree")

    # Default: fallback dialog list = all dialogs
    dialogs_for_rater = dialogs_list

    if not (english_ok and consent and imc_answer == "Strongly disagree"):
        msg = "Please confirm English skills, consent and select 'Strongly disagree' below."
        logger.info(f"[handle_welcome] conditions not met: {msg}")
        return (
            msg,
            None,
            dialogs_for_rater,
            gr.update(visible=True),   # welcome stays
            gr.update(visible=False),  # instructions off
            imc_pass_global,
        )

    # Rater ID
    if not rater_id_input or rater_id_input.strip() == "":
        rater_id = f"R{random.randint(10000, 99999)}"
    else:
        rater_id = rater_id_input.strip()

    logger.info(f"[handle_welcome] effective rater_id={rater_id}")
    logger.info(f"[handle_welcome] imc_pass_global={imc_pass_global}")

    # Slot-based selection
    try:
        dialogs_for_rater = make_main_dialogs_for_rater(n_main=MAX_MAIN_DIALOGS)
        msg = (
            f"Rater ID: **{rater_id}**. A set of dialogues has been assigned to you. "
            f"Please read the instructions on the next page."
        )
        logger.info(f"[handle_welcome] dialogs_for_rater len={len(dialogs_for_rater)} (slot-based)")
    except Exception:
        logger.exception("[handle_welcome] Slot-based assignment failed, falling back to full dialogs_list")
        dialogs_for_rater = dialogs_list
        msg = (
            f"Rater ID: **{rater_id}**. (Note: slot assignment fallback) "
            f"Please read the instructions on the next page."
        )

    return (
        msg,
        rater_id,
        dialogs_for_rater,
        gr.update(visible=False),  # hide welcome
        gr.update(visible=True),   # show instructions
        imc_pass_global,
    )

def start_main_from_instructions(rater_id, dialogs):
    """
    NO PRACTICE: This starts MAIN directly after Instructions.
    """
    logger.info(f"[start_main_from_instructions] rater_id={rater_id!r}")

    if not rater_id:
        rater_id = f"R{random.randint(10000, 99999)}"
        logger.info(f"[start_main_from_instructions] generated fallback rater_id={rater_id}")

    main_index = 0
    start_time = time.time()

    total_dialogs = len(dialogs) if dialogs else 0
    total_main = min(total_dialogs, MAX_MAIN_DIALOGS)

    first_main_html = show_current_dialog(main_index, dialogs) if total_main > 0 else "No dialogs."
    main_progress = main_progress_text(main_index, total_main)

    first_record = dialogs[main_index] if dialogs and total_main > 0 else {}
    main_title = first_record.get("recipe_title") or first_record.get("recipe") or ""

    status = "Instructions confirmed. You will now start the main study."

    return (
        status,                      # status_md
        main_index,                  # main_index_state
        start_time,                  # dialog_start_state
        first_main_html,             # dialog_md (value)
        main_progress,               # progress_md
        main_title,                  # main_title_md
        gr.update(visible=False),    # instructions_group
        gr.update(visible=True),     # main_group
        gr.update(visible=False),    # main_scroll_hint (first dialog off)
    )

In [ ]:
# Gradio Interface CSS

custom_css = """
.dialog-container {
    display: flex;
    flex-direction: column;
    gap: 0.1rem;
    max-width: 48rem;
    margin: 0.25rem auto;
}
.bubble {
    width: 22rem;
    padding: 0.25rem 0.5rem;
    border-radius: 0.75rem;
    font-size: 0.95rem;
    line-height: 1.35;
    word-wrap: break-word;
    overflow-wrap: break-word;
}
.bubble-fat {
    align-self: flex-start;
    background-color: #e3f2fd;
    border: 1px solid #bbdefb;
}
.bubble-carb {
    align-self: flex-end;
    background-color: #fff3e0;
    border: 1px solid #ffe0b2;
}
.bubble-other {
    align-self: center;
    background-color: #eeeeee;
    border: 1px solid #e0e0e0;
}
.speaker {
    font-weight: 600;
    font-size: 0.78rem;
    margin-bottom: 0.02rem;
    opacity: 0.8;
}
.bubble-text { white-space: normal; }

/* Buttons */
button.green-btn {
    background-color: #4caf50 !important;
    color: white !important;
    border-color: #4caf50 !important;
}
button.green-btn[disabled] {
    background-color: #c8e6c9 !important;
    color: #555 !important;
    border-color: #c8e6c9 !important;
}
button.orange-btn {
    background-color: #ff9800 !important;
    color: white !important;
    border-color: #ff9800 !important;
}
button.orange-btn[disabled] {
    background-color: #ffe0b2 !important;
    color: #555 !important;
    border-color: #ffe0b2 !important;
}

/* Hint below comment */
.dialog-hint {
    font-size: 0.9rem;
    color: #555;
    margin-top: 0.25rem;
    margin-bottom: 0.25rem;
}
"""

In [ ]:
with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("## Study A – Dialog Communication Quality", elem_classes=["hint-text", "sticky-header"])

    # States
    rater_id_state = gr.State(value=None)
    dialogs_state = gr.State(value=dialogs_list)
    main_index_state = gr.State(value=0)
    dialog_start_state = gr.State(value=time.time())
    finished_state = gr.State(value=False)
    imc_pass_state = gr.State(value=None)   # global IMC flag

    # -- WELCOME --
    with gr.Group(visible=True) as welcome_group:
        gr.Markdown("### Welcome", elem_classes=["hint-text"])
        rater_id_input = gr.Textbox(label="Rater ID (optional)")
        english_ok_cb = gr.Checkbox(label="I have good English reading skills.")
        consent_cb = gr.Checkbox(label="I agree to participate.")
        imc_radio = gr.Radio(
            ["Strongly disagree", "Disagree", "Neither agree nor disagree", "Agree", "Strongly agree"],
            label="To show that you read carefully, please choose **Strongly disagree**.",
        )

        start_btn = gr.Button("Start", interactive=False, elem_classes=["green-btn"])
        welcome_info = gr.Markdown("Fill in the fields above to start.")

        consent_cb.change(toggle_start_button, inputs=[consent_cb, english_ok_cb, imc_radio], outputs=start_btn)
        english_ok_cb.change(toggle_start_button, inputs=[consent_cb, english_ok_cb, imc_radio], outputs=start_btn)
        imc_radio.change(toggle_start_button, inputs=[consent_cb, english_ok_cb, imc_radio], outputs=start_btn)

    # -- INSTRUCTIONS --
    with gr.Group(visible=False) as instructions_group:
        gr.Markdown(
            """
### Instructions

You will see a series of short dialogues between two nutrition experts.
Each dialogue discusses a recipe from different nutritional perspectives.

Your task is to **evaluate the quality of the communication** in each dialogue.

For every dialogue, please rate the communication on several **7-point rating scales**.
There are no right or wrong answers. I am interested in your personal judgment.

#### Rating scale
You will use a **1 / 4 / 7 principle**:
- **1** = strongly disagree
- **4** = neutral
- **7** = strongly agree

Please consider the **entire dialogue** when rating, not individual statements.
Try to use the full range of the scale where appropriate.

Take your time to read each dialogue carefully before rating.
            """,
            elem_classes=["hint-text"],
        )

        instructions_confirm_cb = gr.Checkbox(
            label="I have read and understood the instructions.",
            value=False,
        )
        instructions_continue_btn = gr.Button(
            "Start main study",
            interactive=False,
            elem_classes=["green-btn"]
        )

        instructions_confirm_cb.change(
            toggle_instructions_button,
            inputs=[instructions_confirm_cb],
            outputs=instructions_continue_btn,
        )

    # -- MAIN --
    with gr.Group(visible=False) as main_group:
        gr.Markdown("### Main dialog rating", elem_classes=["hint-text"])
        progress_md = gr.Markdown("", elem_classes=["hint-text"])
        main_title_md = gr.Markdown("", elem_classes=["title-text"])

        with gr.Row():
            with gr.Column():
                dialog_md = gr.Markdown(
                    "",
                    elem_classes=["dialog-text"],
                    elem_id="main-dialog",
                )

                rating_inputs = []
                for _, lbl in SUBSCALES:
                    rating_inputs.append(
                        gr.Radio(
                            choices=LIKERT_LABELS,
                            label=f"{lbl} (1 = strongly disagree, 4 = neutral, 7 = strongly agree)",
                        )
                    )

                comment_box = gr.Textbox(label="Comment (optional)", lines=2)

                main_scroll_hint = gr.Markdown(
                    "⬆ The **beginning** of the **new** dialogue is above. "
                    "Please scroll up a little before you rate.",
                    elem_classes=["dialog-hint"],
                    visible=False,
                )

                next_btn = gr.Button("Next dialog", elem_classes=["orange-btn"])
                status_md = gr.Markdown("Status will appear here.", elem_classes=["hint-text"])

    # -- END --
    with gr.Group(visible=False) as end_group:
        end_md = gr.Markdown("Thank you for participating.", elem_classes=["hint-text"])

    # --- Welcome Start ---
    start_btn.click(
        handle_welcome,
        inputs=[rater_id_input, english_ok_cb, consent_cb, imc_radio],
        outputs=[
            welcome_info,
            rater_id_state,
            dialogs_state,
            welcome_group,
            instructions_group,
            imc_pass_state,
        ],
    )

    # --- Instructions -> Start MAIN (NO PRACTICE) ---
    instructions_continue_btn.click(
        start_main_from_instructions,
        inputs=[rater_id_state, dialogs_state],
        outputs=[
            status_md,
            main_index_state,
            dialog_start_state,
            dialog_md,
            progress_md,
            main_title_md,
            instructions_group,
            main_group,
            main_scroll_hint,
        ],
    )

    # --- MAIN Next ---
    submit_evt = next_btn.click(
        submit_main_core,
        inputs=[
            *rating_inputs,
            comment_box,
            rater_id_state,
            dialogs_state,
            main_index_state,
            dialog_start_state,
            imc_pass_state,
        ],
        outputs=[
            status_md,
            dialog_md,          # visible
            main_index_state,
            dialog_start_state,
            dialog_md,          # value
            finished_state,
            progress_md,
            main_group,
            end_group,
            end_md,
            main_title_md,
            main_scroll_hint,
        ],
    )

    submit_evt.then(
        reset_main_controls,
        inputs=[],
        outputs=[*rating_inputs, comment_box],
    )

In [ ]:
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://54b4b3a8b34ffe297c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[handle_welcome] rater_id_input='', english_ok=True, consent=True, imc_answer='Strongly disagree'
INFO:studyA:[handle_welcome] rater_id_input='', english_ok=True, consent=True, imc_answer='Strongly disagree'
[handle_welcome] effective rater_id=R20358
INFO:studyA:[handle_welcome] effective rater_id=R20358
[handle_welcome] imc_pass_global=1
INFO:studyA:[handle_welcome] imc_pass_global=1
[make_main_dialogs_for_rater] called with n_main=20
INFO:studyA:[make_main_dialogs_for_rater] called with n_main=20
[make_main_dialogs_for_rater] assign_df shape=(12, 4)
INFO:studyA:[make_main_dialogs_for_rater] assign_df shape=(12, 4)
[make_main_dialogs_for_rater] available shape=(12, 4)
INFO:studyA:[make_main_dialogs_for_rater] available shape=(12, 4)
[make_main_dialogs_for_rater] merged shape=(12, 11), merged columns=['dialog_id', 'recipe_title', 'condition', 'dialog_text', 'is_gold', 'gold_expected_min', 'gold_expected_max', 'is_imc', 'imc_expected_overall', 'remaining_slots', 'recipe_title_assign']
I

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://54b4b3a8b34ffe297c.gradio.live


In [ ]:
# Review results

files = sorted(glob.glob(os.path.join(RESULTS_DIR, "ratings_*.csv")))
print("Found rating files:", len(files))
for f in files:
    df_r = pd.read_csv(f)
    print("\n---", os.path.basename(f), "rows=", len(df_r))
    print("missing overall:", df_r["overall"].isna().sum() if "overall" in df_r.columns else "no overall col")
    print("gold rows:", df_r["is_gold"].sum() if "is_gold" in df_r.columns else "no is_gold col")
    print("imc rows:", df_r["is_imc"].sum() if "is_imc" in df_r.columns else "no is_imc col")
    display(df_r.tail(3))

Found rating files: 3

--- ratings_KK_Mini_Pilot_Test.csv rows= 12
missing overall: 0
gold rows: 2
imc rows: 2


,start_time,timestamp,rater_id,dialog_id,recipe_title,condition,is_gold,gold_expected_min,gold_expected_max,is_imc,...,duration_sec,comment,truthfulness,relevance,clarity,relation_appropriateness,logic_coherence,respect_appreciation,feedback_depth,overall
9,1.767962e+09,1.767962e+09,KK_Mini_Pilot_Test,D046,Best Ever Muffins,bad,0,NaN,NaN,0,...,52.957345,NaN,2,2,2,1,2,1,2,2
10,1.767962e+09,1.767962e+09,KK_Mini_Pilot_Test,D036,No-Bake Energy Bites,good,0,NaN,NaN,0,...,110.273767,NaN,7,6,7,7,6,6,6,7
11,1.767962e+09,1.767962e+09,KK_Mini_Pilot_Test,D039,Grandma's Apple Dumplings,good,0,NaN,NaN,0,...,112.765756,NaN,7,7,6,7,7,7,6,7



--- ratings_R20358.csv rows= 12
missing overall: 0
gold rows: 2
imc rows: 2


,start_time,timestamp,rater_id,dialog_id,recipe_title,condition,is_gold,gold_expected_min,gold_expected_max,is_imc,...,duration_sec,comment,truthfulness,relevance,clarity,relation_appropriateness,logic_coherence,respect_appreciation,feedback_depth,overall
9,1.767979e+09,1.767980e+09,R20358,D039,Grandma's Apple Dumplings,good,0,NaN,NaN,0,...,187.927332,NaN,6,6,5,7,6,7,5,6
10,1.767980e+09,1.767980e+09,R20358,D001,Moong Dal,good,1,6.0,7.0,0,...,153.866946,NaN,5,4,2,7,3,6,5,4
11,1.767980e+09,1.767980e+09,R20358,D016,Strawberry Rhubarb Coffee Cake,good,0,NaN,NaN,0,...,184.863021,Best conversation! I like the end-statement - ...,7,7,7,7,7,7,7,7



--- ratings_R37634.csv rows= 12
missing overall: 0
gold rows: 2
imc rows: 2


,start_time,timestamp,rater_id,dialog_id,recipe_title,condition,is_gold,gold_expected_min,gold_expected_max,is_imc,...,duration_sec,comment,truthfulness,relevance,clarity,relation_appropriateness,logic_coherence,respect_appreciation,feedback_depth,overall
9,1.767979e+09,1.767980e+09,R37634,D001,Moong Dal,good,1,6.0,7.0,0,...,212.403536,NaN,5,5,5,7,6,6,6,6
10,1.767980e+09,1.767980e+09,R37634,D039,Grandma's Apple Dumplings,good,0,NaN,NaN,0,...,163.585295,NaN,5,5,6,6,5,6,5,5
11,1.767980e+09,1.767980e+09,R37634,D036,No-Bake Energy Bites,good,0,NaN,NaN,0,...,110.359955,NaN,7,6,7,7,7,7,7,7
